# CUSTOMER CHURN & REVENUE INTELLIGENCE
## Part 2: Behavioral Customer Segmentation & RFM Clustering
**Author:** Senior Data Analyst & BI Developer  
**Algorithm:** K-Means Clustering & Principal Component Analysis (PCA)  
**Objective:** Group customers into homogeneous behavioral clusters based on tenure, platform usage, spending habits, and service escalations to guide personalized retention campaigns.

---
### Table of Contents:
1. Business Context & Strategic Segmentation Objectives
2. Feature Selection & Preprocessing
3. Optimal Cluster Selection (Elbow Method & Silhouette Scores)
4. K-Means Model Execution (K=4)
5. Behavioral Cluster Profiling & Statistical Summaries
6. Dimensionality Reduction & 2D PCA Visualization
7. Revenue & Risk Exposure by Segment
8. Strategic Retention Playbooks for Each Segment


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../dataset/customer_churn_cleaned.csv')
print(f"Loaded {len(df):,} records for behavioral customer segmentation.")


## 1. Feature Selection & Normalization
We select four continuous behavioral drivers:
- `Tenure_Months`: Customer longevity and brand loyalty.
- `Monthly_Charges`: Ongoing monetary commitment (ARPU).
- `Usage_Hours`: Platform engagement and digital utilization.
- `Support_Tickets`: Friction and service dissatisfaction indicator.

In [ ]:
features = ['Tenure_Months', 'Monthly_Charges', 'Usage_Hours', 'Support_Tickets']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled to zero mean and unit variance:")
pd.DataFrame(X_scaled, columns=features).describe()


## 2. Optimal Cluster Determination
We evaluate K=2 through K=7 using Inertia (Elbow Method) and Silhouette Analysis.

In [ ]:
inertias = []
sil_scores = []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

eval_df = pd.DataFrame({
    'Clusters (K)': list(k_range),
    'Inertia (SSE)': inertias,
    'Silhouette Score': sil_scores
})
eval_df


### Model Choice: K=4
K=4 provides the optimal balance between high silhouette separability and actionable, commercially intuitive SaaS personas.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Assign meaningful SaaS strategic personas
persona_mapping = {
    0: 'Tier 1: High-Spender Heavy Streamers',
    1: 'Tier 2: Established Loyalists',
    2: 'Tier 3: Budget Disengaged Newbies',
    3: 'Tier 4: Moderate Standard Users'
}
df['Customer_Segment'] = df['Cluster'].map(persona_mapping)

print("Cluster assignment distribution:")
df['Customer_Segment'].value_counts()


## 3. Segment Profiling & Business Characteristics
Let's inspect the mean metrics for each customer segment.

In [ ]:
segment_profile = df.groupby('Customer_Segment').agg(
    Customer_Count=('CustomerID', 'count'),
    Avg_Tenure_Months=('Tenure_Months', 'mean'),
    Avg_Monthly_Charges=('Monthly_Charges', 'mean'),
    Avg_Usage_Hours=('Usage_Hours', 'mean'),
    Avg_Support_Tickets=('Support_Tickets', 'mean'),
    Avg_CLV=('Total_Charges', 'mean'),
    Churn_Rate=('Churn_Status', lambda s: (s == 'Yes').mean() * 100),
    Total_MRR=('Monthly_Charges', 'sum')
).reset_index()

segment_profile['%_of_Base'] = (segment_profile['Customer_Count'] / len(df) * 100).round(1)
segment_profile.sort_values(by='Total_MRR', ascending=False)


## 4. 2D Dimensionality Reduction (PCA Visualization)

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['PCA1'] = pca_coords[:, 0]
df['PCA2'] = pca_coords[:, 1]

print(f"Explained Variance Ratio: {pca.explained_variance_ratio_ * 100}%")
print(f"Total Variance Captured: {sum(pca.explained_variance_ratio_)*100:.2f}%")


## 5. Strategic Playbooks for Each Segment
1. **Tier 1: High-Spender Heavy Streamers:**
   - *Characteristics:* High ARPU ($95+), high bandwidth usage (100+ hrs), moderate ticket escalations.
   - *Strategy:* Proactive VIP technical support, network quality SLA guarantees, loyalty discounts for multi-year contract renewals.
2. **Tier 2: Established Loyalists:**
   - *Characteristics:* High tenure (45+ months), two-year contracts, low support tickets, very low churn rate (<5%).
   - *Strategy:* Advocacy programs, cross-selling enterprise add-ons, referral incentives.
3. **Tier 3: Budget Disengaged Newbies:**
   - *Characteristics:* Low tenure (0-6 months), low usage (<25 hrs), basic subscription plan, highest churn rate (>55%).
   - *Strategy:* Digital onboarding walkthroughs, feature adoption nudges, automated check-ins at Day 14 and Day 45.
4. **Tier 4: Moderate Standard Users:**
   - *Characteristics:* Balanced usage and billing, standard tier.
   - *Strategy:* Bundle upgrades, payment auto-pay migration incentives ($5/mo bill credit for switching from electronic check to automatic credit card).
